<a href="https://colab.research.google.com/github/rajvi-m/data-analysis-practice/blob/main/capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mysql-connector-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 63.3 MB/s eta 0:00:00


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import mysql.connector as m

df=pd.read_csv('database.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date'])  #drop rows where date couldn't be parsed
df['Date'] = df['Date'].dt.date

#sql for storing relevant data in mysql tables from csv
conn = m.connect(host="localhost",user="root",password="1234",database="my_database")
mycur=conn.cursor()
query="CREATE TABLE if not exists earthquakes (date DATE,latitude FLOAT, longitude FLOAT,magnitude FLOAT,depth FLOAT,location VARCHAR(255))"
mycur.execute(query)
print("Table created")
mycur.execute("DELETE FROM earthquakes")
conn.commit()
#inserting required rows from pandas dataframe to mysql tables
for _, row in df.iterrows():
    mycur.execute("""INSERT INTO earthquakes (date,latitude,longitude, magnitude, depth, location) VALUES (%s, %s, %s, %s,%s,%s)""", (row['Date'],row['Latitude'],row['Longitude'], row['Magnitude'], row['Depth'], row.get('Location Source','')))

# 1. earthquake group by years
mycur.execute("SELECT YEAR(date) AS year,COUNT(*) AS total_earthquakes FROM earthquakes GROUP BY YEAR(date) ORDER BY year;")
print("Earthquakes grouped by year")
rec=mycur.fetchall()
for row in rec:
    print("year:",row[0],"number of earthquakes:",row[1])
#graph
years=[row[0] for row in rec]
count=[row[1] for row in rec]
plt.figure(figsize=(12,6))
plt.bar(years,count,color='green')
plt.xlabel('Year')
plt.ylabel('Number of Earthquakes')
plt.title('Earthquakes per Year')
plt.show()

# 1. earthquake group by depth
mycur.execute("SELECT CASE WHEN depth <= 100 THEN 'Shallow (0–100 km)' WHEN depth <= 300 THEN 'Intermediate (101–300 km)' ELSE 'Deep (>300 km)' END AS depth_range, COUNT(*) AS total FROM earthquakes Group by depth_range")
print("\nEarthquakes grouped by depth")
rec= mycur.fetchall()
for row in rec:
    print(row[0],"number of earthquakes:",row[1])
#graph pie chart of depth range of earthquakes
depth=[row[0] for row in rec]
count=[row[1] for row in rec]
plt.pie(count,startangle=90,labels=depth,colors=['rosybrown', 'indianred', 'maroon'],autopct='%1.1f%%' )
plt.legend(loc='lower left')
plt.title('Earthquakes by Depth Range')
plt.show()

# 2. Top 5 most powerful earthquake
mycur.execute("select * from earthquakes order by magnitude desc limit 5")
print("\nTop 5 most powerful earthquakes")
data=mycur.fetchall()
for r in data:
    print(r)
# 2. top 5 deepest earthquake
mycur.execute("select * from earthquakes order by depth desc limit 5")
print("\nTop 5 deepest earthquakes")
data=mycur.fetchall()
for r in data:
    print(r)

#fetching all the rows into pandas dataframe
query = "SELECT * FROM earthquakes"
mycur.execute(query)
rec=mycur.fetchall()
df = pd.read_sql(query, conn)
#graph magnitude vs depth
magnitudes = [row[3] for row in rec]
depths = [row[4] for row in rec]
plt.scatter(depths,magnitudes,color='teal',s=10)
plt.title('Magnitude vs. Depth of Earthquakes')
plt.xlabel('Depth (km)')
plt.ylabel('Magnitude')
plt.show()

mycur.execute("SELECT YEAR(date) AS year, AVG(magnitude) AS avg_magnitude FROM earthquakes GROUP BY YEAR(date) ORDER BY year;")
rec=mycur.fetchall()
year=[row[0] for row in rec]
mag=[row[1] for row in rec]
plt.plot(year,mag,color='blue')
plt.xlabel('Year')
plt.ylabel('Average magnitude')
plt.title('Average magnitude per year')
plt.show()
#graph earthquake analysis by hemisphere
mycur.execute("SELECT CASE WHEN latitude >= 0 THEN 'Northern Hemisphere' ELSE 'Southern Hemisphere' END AS hemisphere, COUNT(*) AS total FROM earthquakes GROUP BY hemisphere")
rec = mycur.fetchall()
labels = [row[0] for row in rec]
counts = [row[1] for row in rec]
colors = ['lightblue', 'lightcoral']
plt.figure(figsize=(6,6))
plt.pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Earthquakes by Hemisphere')
plt.show()
mycur.close()
conn.close()

# 3. Mean, Max and min Magnitude and depth of earthquake
print("Mean Magnitude:", df['magnitude'].mean())
print("Max Magnitude:", df['magnitude'].max())
print("Min Magnitude:", df['magnitude'].min())

print("Mean Depth:", df['depth'].mean())
print("Max Depth:", df['depth'].max())
print("Min Depth:", df['depth'].min())





InterfaceError: 2003: Can't connect to MySQL server on 'localhost:3306' (Errno 111: Connection refused)